In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/latest.pt
/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_val.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_val.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/test.json
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_train_f.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_val_f.bin


In [2]:
import os
import re
import gc
import json
import time
import math
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tokenizers import Tokenizer
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download

# =====================================================================
# 1. Download Dataset & Initialize Tokenizer
# =====================================================================
TRAIN_RAW_PATH = "/kaggle/working/Phase_6_Summ/Train/train.json"
TEST_RAW_PATH  = "/kaggle/working/Phase_6_Summ/Test/test.json"

if not os.path.exists(TRAIN_RAW_PATH):
    print("Downloading Phase_6_Summ dataset from Hugging Face...")
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    snapshot_download(
        repo_id="Exploration-Lab/CS-781-Capstone",
        repo_type="dataset",
        token=HF_TOKEN,
        allow_patterns=["Phase_6_Summ/**"],
        local_dir="/kaggle/working",
    )

TOKENIZER_PATH = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"

class Phase1Tokenizer:
    def __init__(self, path=TOKENIZER_PATH):
        self.tok = Tokenizer.from_file(path)
        self.pad_id = 0
        self.bos_id = 1
        self.eos_id = 2
        self.vocab_size = self.tok.get_vocab_size()

    def encode(self, text: str):
        return self.tok.encode(text, add_special_tokens=False).ids

    def decode(self, ids):
        if torch.is_tensor(ids):
            ids = ids.tolist()
        clean = [int(i) for i in ids if int(i) not in (self.pad_id, self.bos_id, self.eos_id) and int(i) >= 0]
        return self.tok.decode(clean, skip_special_tokens=True).strip()

tokenizer = Phase1Tokenizer()

# =====================================================================
# 2. Advanced Legal Condensation & Post-Processing
# =====================================================================
LEGAL_KEYWORDS = {
    "held", "court", "appeal", "dismissed", "allowed", "section", "article",
    "act", "petition", "petitioner", "respondent", "judgment", "order",
    "tribunal", "issue", "counsel", "quashed", "convicted", "acquitted",
    "bench", "supreme", "high", "suit", "decree", "plaintiff", "defendant"
}

def cue_salience_condense(all_ids: list, max_tokens: int = 1024):
    if len(all_ids) <= max_tokens:
        return all_ids
    h_budget = int(max_tokens * 0.30)
    t_budget = int(max_tokens * 0.40)
    m_budget = max_tokens - h_budget - t_budget

    head_ids = all_ids[:h_budget]
    tail_ids = all_ids[-t_budget:]
    mid_text = tokenizer.decode(all_ids[h_budget:-t_budget])
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+|\n+', mid_text) if len(s.strip()) > 30]

    if not sentences:
        mid_start = (len(all_ids) - m_budget) // 2
        return head_ids + all_ids[mid_start:mid_start + m_budget] + tail_ids

    scored = []
    for idx, s in enumerate(sentences):
        words = re.findall(r'\w+', s.lower())
        if not words:
            continue
        hits = sum(1 for w in words if w in LEGAL_KEYWORDS)
        digits = sum(1 for w in words if w.isdigit())
        score = (hits * 2.5 + digits * 1.5) / (len(words) ** 0.5)
        scored.append((score, idx, s))

    scored.sort(key=lambda x: x[0], reverse=True)
    selected, used = [], 0
    for score, idx, s in scored:
        s_ids = tokenizer.encode(s)
        if used + len(s_ids) <= m_budget:
            selected.append((idx, s_ids))
            used += len(s_ids)
        if used >= m_budget - 15:
            break

    selected.sort(key=lambda x: x[0])
    mid_ids = [tid for _, s_ids in selected for tid in s_ids]
    return (head_ids + mid_ids + tail_ids)[:max_tokens]

def clean_legal_summary_postprocess(text: str) -> str:
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return text
    if text[-1] not in ".!?;\"')":
        last_period = max(text.rfind(". "), text.rfind(".\n"), text.rfind("; "))
        if last_period != -1 and last_period >= int(len(text) * 0.80):
            text = text[:last_period + 1].strip()
        else:
            text = text + "."
    return text

# =====================================================================
# 3. Model Architecture (Arch4 Hybrid GQA - Best Performing Base)
# =====================================================================
def apply_rope(x, cos, sin):
    d = x.shape[-1] // 2
    x1, x2 = x[..., :d], x[..., d:]
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, seq_len, device, dtype):
        t = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        return freqs.cos().to(dtype)[None, None, :, :], freqs.sin().to(dtype)[None, None, :, :]

class Arch4GQAAttention(nn.Module):
    def __init__(self, d_model=1024, n_heads=16, n_kv_heads=8):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.kv_dim = n_kv_heads * self.head_dim
        self.qkv_proj = nn.Linear(d_model, d_model + 2 * self.kv_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.rope = RotaryEmbedding(self.head_dim)

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.qkv_proj(x)
        q, k, v = torch.split(qkv, [C, self.kv_dim, self.kv_dim], dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        cos, sin = self.rope(T, x.device, x.dtype)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        rep = self.n_heads // self.n_kv_heads
        k = k.repeat_interleave(rep, dim=1)
        v = v.repeat_interleave(rep, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T, C))

class Arch4GatedFFN(nn.Module):
    def __init__(self, d_model=1024, d_ff=3328):
        super().__init__()
        self.in_proj = nn.Linear(d_model, 2 * d_ff, bias=False)
        self.out_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        gate, up = self.in_proj(x).chunk(2, dim=-1)
        return self.out_proj(F.silu(gate) * up)

class Arch4Layer(nn.Module):
    def __init__(self, d_model=1024, n_heads=16, n_kv_heads=8, d_ff=3328):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = Arch4GQAAttention(d_model, n_heads, n_kv_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = Arch4GatedFFN(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.ffn(self.norm2(x))

class Arch4HybridGQAModel(nn.Module):
    def __init__(self, vocab_size=40000, d_model=1024, n_layers=12, n_heads=16, n_kv_heads=8, d_ff=3328):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([Arch4Layer(d_model, n_heads, n_kv_heads, d_ff) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids, attention_mask=None):
        x = self.token_embedding(input_ids)
        for layer in self.layers:
            x = layer(x)
        return self.lm_head(self.final_norm(x))

# =====================================================================
# 4. Compulsory LORMA (Low-Rank Multiplicative Adaptation)
# =====================================================================
class LORMALinear(nn.Module):
    def __init__(self, base_layer: nn.Linear, rank: int = 32, alpha: float = 64.0, dropout: float = 0.05):
        super().__init__()
        self.base_layer = base_layer
        for p in self.base_layer.parameters():
            p.requires_grad = False
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(dropout)
        out_dim = base_layer.out_features
        self.lorma_A = nn.Parameter(torch.empty(rank, out_dim))
        self.lorma_B = nn.Parameter(torch.zeros(out_dim, rank))
        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))

    def forward(self, x):
        base_out = self.base_layer(x)
        mod = (self.dropout(base_out) @ self.lorma_A.T) @ self.lorma_B.T
        return base_out + mod * self.scaling

def inject_lorma(model: nn.Module, rank=32):
    for p in model.parameters():
        p.requires_grad = False
    skip_names = ("token_emb", "token_embedding", "pos_emb", "lm_head")
    for name, module in list(model.named_modules()):
        if isinstance(module, nn.Linear) and not any(s in name for s in skip_names):
            parent_name, child_name = name.rsplit(".", 1) if "." in name else ("", name)
            parent = model.get_submodule(parent_name) if parent_name else model
            setattr(parent, child_name, LORMALinear(module, rank=rank))
    for m in model.modules():
        if isinstance(m, nn.LayerNorm):
            for p in m.parameters():
                p.requires_grad = True
    return model

# =====================================================================
# 5. Dataset & Training Pipeline
with open(TRAIN_RAW_PATH, "r", encoding="utf-8") as f:
    raw_train = json.load(f)
with open(TEST_RAW_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

train_clean = [
    x for x in raw_train
    if len(x["summary"].strip().split()) >= 10 and len(x["document"].strip().split()) >= 20
]

print(f"Pre-tokenizing {len(train_clean)} train and {len(test_data)} test docs...")
for ex in train_clean:
    ex["doc_ids"] = tokenizer.encode(ex["document"])
    ex["sum_ids"] = tokenizer.encode(ex["summary"])
for ex in test_data:
    ex["doc_ids"] = tokenizer.encode(ex["document"])

class ProDataset(Dataset):
    def __init__(self, examples, max_src_len=1024, max_tgt_len=540):
        self.examples = examples
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len
        self.sep_ids = tokenizer.encode("\n\nSummary:\n")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        src_ids = cue_salience_condense(ex["doc_ids"], self.max_src_len)
        sum_ids = ex["sum_ids"][:self.max_tgt_len - 1] + [tokenizer.eos_id]
        prompt_ids = [tokenizer.bos_id] + src_ids + self.sep_ids
        full_ids = prompt_ids + sum_ids
        labels = ([-100] * len(prompt_ids)) + sum_ids
        return {
            "input_ids": torch.tensor(full_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

def collate_fn(batch):
    input_ids = nn.utils.rnn.pad_sequence([b["input_ids"] for b in batch], batch_first=True, padding_value=0)
    labels = nn.utils.rnn.pad_sequence([b["labels"] for b in batch], batch_first=True, padding_value=-100)
    return {"input_ids": input_ids, "labels": labels}

# Load Arch4 Checkpoint
ckpt_path = "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt"
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
sd = {k.replace("_orig_mod.", ""): v for k, v in ckpt["model"].items()}
model = Arch4HybridGQAModel()
model.load_state_dict(sd, strict=False)
model = inject_lorma(model, rank=32).cuda()

train_loader = DataLoader(ProDataset(train_clean), batch_size=2, shuffle=True, collate_fn=collate_fn)
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=4e-4, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler()

print("Training Arch4 LORMA Pro for 2,000 micro-steps...")
model.train()
global_step = 0
for epoch in range(1):
    for batch in train_loader:
        global_step += 1
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16):
            logits = model(batch["input_ids"].cuda())
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = batch["labels"][:, 1:].contiguous().cuda()
            loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), ignore_index=-100) / 8

        scaler.scale(loss).backward()
        if global_step % 8 == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        if global_step % 400 == 0:
            print(f"  Step {global_step}/2000 | Loss: {loss.item()*8:.4f}")

        if global_step >= 2000:
            break

# =====================================================================
# 6. Generate Codabench Submission (`prediction.json` at Root)
# =====================================================================
@torch.no_grad()
def generate_pro_test(model, test_examples):
    model.eval()
    results = []
    sep_ids = tokenizer.encode("\n\nSummary:\n")
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    for idx, ex in enumerate(test_examples):
        src_ids = cue_salience_condense(ex["doc_ids"], 1024)
        input_ids = [tokenizer.bos_id] + src_ids + sep_ids
        generated = []

        for _ in range(480):  # Full-length generation capacity
            inp_tensor = torch.tensor([input_ids], dtype=torch.long, device="cuda")
            with torch.amp.autocast(device_type="cuda", dtype=amp_dtype):
                next_logits = model(inp_tensor)[0, -1, :].float()

            for p_id in set(generated[-64:]):
                next_logits[p_id] /= 1.2
            if len(generated) >= 2:
                p2 = (generated[-2], generated[-1])
                for i in range(len(generated) - 2):
                    if (generated[i], generated[i+1]) == p2:
                        next_logits[generated[i+2]] = -1e9

            next_id = int(torch.argmax(next_logits).item())
            if next_id == tokenizer.eos_id and len(generated) >= 60:
                break
            generated.append(next_id)
            input_ids.append(next_id)

        summary_text = clean_legal_summary_postprocess(tokenizer.decode(generated))
        if not summary_text:
            summary_text = clean_legal_summary_postprocess(ex["document"][:600])
        results.append({"id": str(ex["id"]), "summary": summary_text})
    return results

print("Generating 100 test summaries with Arch4 LORMA Pro...")
preds = generate_pro_test(model, test_data)

# Package strictly as prediction.json at root of submission.zip
tmp_json = "/kaggle/working/prediction.json"
with open(tmp_json, "w", encoding="utf-8") as f:
    json.dump(preds, f, indent=2, ensure_ascii=False)

zip_path = "/kaggle/working/submission.zip"
with zipfile.ZipFile(zip_path, mode="w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(tmp_json, arcname="prediction.json")

with zipfile.ZipFile(zip_path, mode="r") as zf:
    assert zf.namelist() == ["prediction.json"]

print(f"✅ Successfully created {zip_path} with prediction.json at root!")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Pre-tokenizing 7028 train and 100 test docs...


/tmp/ipykernel_23/2587281833.py:292: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Training Arch4 LORMA Pro for 2,000 micro-steps...
  Step 400/2000 | Loss: 2.5496
  Step 800/2000 | Loss: 2.7292
  Step 1200/2000 | Loss: 2.7007
  Step 1600/2000 | Loss: 2.8674
  Step 2000/2000 | Loss: 2.7027
Generating 100 test summaries with Arch4 LORMA Pro...
✅ Successfully created /kaggle/working/submission.zip with prediction.json at root!
